# Step 2: Scale with SageMaker AI jobs

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.6.0</strong> and the <strong>SageMaker Python SDK version 3.27.0</strong>.
</div>

In this step you move data processing and model training into [SageMaker Docker containers](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html) and use [SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/index.html) to interact with SageMaker.

||||
|---|---|---|
|1. |Experiment in a notebook ||
|2. |Scale with SageMaker AI jobs and SageMaker SDK |**<<<< YOU ARE HERE**|
|3. |Operationalize with ML pipeline, model registry ||
|4. |Deploy the model and add data and model monitoring ||
|5. |Automate model deployment with a SageMaker project ||

SageMaker makes use of Docker containers to enable developers to process data, train and deploy models. Containers allow developers and data scientists to package software into standardized units that run consistently on any platform that supports Docker. Containers ensure that code, runtime, system tools, system libraries, and settings are all in the same place, isolating them from the execution environment. It guarantees a consistent runtime experience regardless of where a container is being run.

SageMaker also provides pre-built containers with popular data processing frameworks and ML algorithms, and all SageMaker built-in algorithms are delivered as Docker containers.

The data processing and training code lives in reusable Python functions in the [`pipeline_steps`](pipeline_steps/) folder. You use the same functions in this notebook and in the ML pipeline in step 3. In this notebook you run them in two ways:
1. **`@remote` decorator**: the simplest path. SageMaker runs a Python function as a SageMaker job and returns its result to the notebook
2. **`ModelTrainer` in script mode**: full control over the training job. SageMaker runs a Python file as the entry script of the training container

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"This workshop needs Python 3.12, but the kernel runs Python {sys.version_info.major}.{sys.version_info.minor}. "
    "Run ./setup-py312.sh in a terminal, refresh the browser, and select the Python 3.12 kernel. "
    "See 00-start-here.ipynb for the steps."
)

Load the stored variables from previous notebooks.

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
import warnings
import os
import time
import inspect
from html import escape
from IPython.display import HTML
import tempfile
import shutil
import boto3
import mlflow
import sagemaker_mlflow
from mlflow import MlflowClient
import pandas as pd
import xgboost as xgb
import numpy as np
from numpy import loadtxt
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn import metrics
import matplotlib.pyplot as plt

from sagemaker.core.utils.utils import configure_logging
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.s3.client import s3_path_join
from sagemaker.core.shapes import OutputDataConfig
from sagemaker.core.resources import ModelPackage
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.remote_function.client import remote
from sagemaker.core.remote_function.custom_file_filter import CustomFileFilter
from sagemaker.train.configs import InputData, SourceCode, Compute
from sagemaker.train import ModelTrainer
from sagemaker.train.model_trainer import Mode

# Reusable functions shared with the ML pipeline in notebook 03
from pipeline_steps.preprocess import preprocess
from pipeline_steps.train_fn import train

In [ ]:
warnings.filterwarnings('ignore')
configure_logging('INFO')

## Prepare an MLflow experiment
Re-use the existing experiment we created in the previous notebook. You're going to track new runs in the same experiment.
You can also create a new experiment to track runs in this notebook – in this case just uncomment the following code cell.

In [ ]:
# The experiment name is restored from %store (created in notebook 01).
# Fall back to the same date-suffixed name if this notebook is run standalone.
if "experiment_name" not in dir():
    import time
    experiment_name = f"from-idea-to-prod-experiment-{time.strftime('%d%m%y', time.gmtime())}"
    %store experiment_name

print(f"Using MLflow experiment: {experiment_name}")

In [ ]:
# MLflow prints "View run ... at: https://mlflow.sagemaker.<region>.app.aws/..." when a run
# ends. That host is the SageMaker MLflow *tracking API* endpoint, not the UI, so the links
# do not open. Suppress them; the notebook shows working presigned UI links instead.
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"
mlflow.set_tracking_uri(mlflow_arn)
experiment = mlflow.set_experiment(experiment_name=experiment_name)
print(f'Using MLflow server: {mlflow_arn}')

## Reusable functions for processing and training
The folder [`pipeline_steps`](pipeline_steps/) contains the data processing and model training code from notebook 01 as plain Python functions:
- `preprocess()` in [`pipeline_steps/preprocess.py`](pipeline_steps/preprocess.py) runs the feature engineering, splits the data, and writes the train, validation, test, and baseline datasets to S3
- `train()` in [`pipeline_steps/train_fn.py`](pipeline_steps/train_fn.py) trains the XGBoost model and logs it to MLflow

Both functions track their parameters, metrics, and artifacts in MLflow. They read the MLflow tracking server and experiment from the environment variables `MLFLOW_TRACKING_URI` and `MLFLOW_EXPERIMENT_NAME`, so the same code works in the notebook and in a SageMaker job without changes. In step 3 you build an ML pipeline from the same functions.

The next cell defines a helper to show the source code of a function in the notebook.

In [ ]:
def source_details(function):
    """Return a collapsed, HTML-escaped view of a function's source code."""
    qualified_name = f"{function.__module__}.{function.__name__}"
    signature = escape(str(inspect.signature(function)))
    source = escape(inspect.getsource(function))
    return HTML(
        f"""
        <details style="margin: 0.5rem 0 1rem; border: 1px solid var(--jp-border-color2, #ddd); border-radius: 6px; padding: 0.6rem 0.8rem;">
          <summary style="cursor: pointer; font-weight: 600;">
            View source: <code>{escape(qualified_name)}{signature}</code>
          </summary>
          <pre style="margin-top: 0.8rem; max-height: 32rem; overflow: auto; white-space: pre;"><code>{source}</code></pre>
        </details>
        """
    )

### Settings for SageMaker jobs
The [`@remote` decorator](https://docs.aws.amazon.com/sagemaker/latest/dg/train-remote-decorator.html) serializes a Python function and its arguments, runs the function as a SageMaker training job, and returns the result to the notebook. The job runs on the custom BYOC image you built in notebook 00, which already contains all dependencies.

`include_local_workdir=True` uploads the workshop folder to the job, so that the job can import the `pipeline_steps` module. The file filter excludes data, images, and notebooks from the upload.

In [ ]:
region = Session().boto_region_name
output_s3_prefix = s3_path_join("s3://", bucket_name, bucket_prefix)

# MLflow reads the first two variables natively. pipeline_steps/runtime_utils.py logs CONTAINER_IMAGE as a run tag.
job_environment = {
    "MLFLOW_TRACKING_URI": mlflow_arn,
    "MLFLOW_EXPERIMENT_NAME": experiment_name,
    "CONTAINER_IMAGE": xgboost_image,
    "USER": user_profile_name,
    "REGION": region,
}

remote_settings = {
    "instance_type": "ml.m5.xlarge",
    "role": sm_role,
    "image_uri": xgboost_image,
    "include_local_workdir": True,
    "custom_file_filter": CustomFileFilter(
        ignore_name_patterns=[
            "data", "tmp", "img", "design", "assignments", "additional-topics",
            "deployment-seed-code", "processing", "training", "*.ipynb", "__pycache__", ".git",
        ]
    ),
    "environment_variables": job_environment,
    "keep_alive_period_in_seconds": 3600,
}

# The functions read the same variables when you call them in the notebook.
# CONTAINER_IMAGE=local marks these runs as local in MLflow.
os.environ.update({**job_environment, "CONTAINER_IMAGE": "local"})

print(f"Image:  {xgboost_image}")
print(f"Output: {output_s3_prefix}")

---

## Process data
### Inspect `preprocess()`
Expand the panel to see the function. It reuses the feature engineering from notebook 01 in `engineer_features()`, splits the data, and writes the datasets as header-less CSV files with the label in the first column. It also writes the whole dataset without the label column as a baseline dataset, which you need for model monitoring in step 4.

In [ ]:
display(source_details(preprocess))

### Validate `preprocess()` locally
Call the function in the notebook with the local dataset and a local output folder. This is a quick check before you start a SageMaker job.

In [ ]:
local_output_prefix = os.path.abspath("tmp/local-validation")
for split in ["train", "validation", "test", "baseline"]:
    os.makedirs(os.path.join(local_output_prefix, split), exist_ok=True)

local_preprocess_result = preprocess(
    input_data_s3_path=dataset_file_local_path,
    output_s3_prefix=local_output_prefix,
    pipeline_run_name=name_from_base("notebook-02-local", short=True),
)
local_preprocess_result

### Run `preprocess()` on SageMaker
Wrap the same function with `remote()` and call it with the dataset in S3. The function runs as a SageMaker training job and returns the same dictionary as the local call.

The function also creates a parent MLflow run. You pass its ID to the training functions below, so that all runs of this notebook are grouped under one parent run in MLflow.

In [ ]:
remote_preprocess = remote(**remote_settings, job_name_prefix="from-idea-to-prod-preprocess")(preprocess)

preprocess_result = remote_preprocess(
    input_data_s3_path=s3_data_uri,
    output_s3_prefix=output_s3_prefix,
    pipeline_run_name=name_from_base("notebook-02", short=True),
)
preprocess_result

Store the S3 locations of the datasets for the next notebooks.

In [ ]:
train_s3_url = preprocess_result["train_data"].rsplit("/", 1)[0]
validation_s3_url = preprocess_result["validation_data"].rsplit("/", 1)[0]
test_s3_url = preprocess_result["test_x_data"].rsplit("/", 1)[0]
baseline_s3_url = preprocess_result["baseline_data"].rsplit("/", 1)[0]

%store train_s3_url
%store validation_s3_url
%store test_s3_url
%store baseline_s3_url

Display the first few rows of the training data for a quick sanity check. The first column is the label.

In [ ]:
pd.read_csv(preprocess_result["train_data"], header=None).head()

---

## Model training with SageMaker training jobs
You can follow the same approach as with processing and now run the model training as a [SageMaker training job](https://docs.aws.amazon.com/sagemaker/latest/dg/train-model.html).

SageMaker AI offers four ways to train a model. This workshop uses its own container, which you built in notebook 00.

![SageMaker AI training options: built-in algorithms, script mode, your own container, and AWS Marketplace](img/sagemaker-training-options.svg)

Set the instance type and instance count for the training jobs.

In [ ]:
train_instance_count = 1
train_instance_type = "ml.r5.xlarge"

Set the hyperparameters. `train()` accepts them as keyword arguments.

In [ ]:
hyperparams = {
    'num_round': 50,
    'max_depth': 3,
    'eta': 0.5,
    'alpha': 2.5,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'early_stopping_rounds': 10,
    'verbosity': 1
}

### Inspect `train()`

In [ ]:
display(source_details(train))

### Validate `train()` locally
Train on the datasets that the SageMaker job wrote to S3.

In [ ]:
local_training_result = train(
    train_data_s3_path=preprocess_result["train_data"],
    validation_data_s3_path=preprocess_result["validation_data"],
    output_s3_prefix=local_output_prefix,
    pipeline_run_id=preprocess_result["pipeline_run_id"],
    **hyperparams,
)
print(f"Local validation AUC: {local_training_result['validation_auc']:.4f}")

### Train with the `@remote` decorator
Wrap `train()` with `remote()`, the same way you did for `preprocess()`. This is the quickest way to move a local Python function to SageMaker managed infrastructure.

In [ ]:
remote_train = remote(
    **{**remote_settings, "instance_type": train_instance_type},
    job_name_prefix="from-idea-to-prod-remote-train",
)(train)

remote_training_result = remote_train(
    train_data_s3_path=preprocess_result["train_data"],
    validation_data_s3_path=preprocess_result["validation_data"],
    output_s3_prefix=output_s3_prefix,
    pipeline_run_id=preprocess_result["pipeline_run_id"],
    **hyperparams,
)
print(f"Remote validation AUC: {remote_training_result['validation_auc']:.4f}")

---

### Train with `ModelTrainer` in script mode
The [`ModelTrainer`](https://sagemaker.readthedocs.io/en/stable/api/training/model_trainer.html) class gives you full control over the training job configuration: the training image, the entry script, the input channels, the hyperparameters, and the output location. Use it when you need a specific training container, a SageMaker built-in image or your own, or when you run training jobs at scale.

Instead of a Python function, `ModelTrainer` runs a Python file as the entry script in the container. You don't need a separate training script: you use the same file [`pipeline_steps/train_fn.py`](pipeline_steps/train_fn.py). When SageMaker runs the file, its `main()` function at the end of the file reads the hyperparameters and the input data that SageMaker passes to the script, and calls the same `train()` function.

Define the input channels. SageMaker downloads the content of each S3 location to `/opt/ml/input/data/<channel_name>` in the container.

In [ ]:
training_input_data = [
    InputData(channel_name="train", data_source=train_s3_url),
    InputData(channel_name="validation", data_source=validation_s3_url),
]

output_s3_url = f"s3://{bucket_name}/{bucket_prefix}/output"
%store output_s3_url

Now define the `ModelTrainer` with the BYOC image, the source code, the compute specifications, the hyperparameters, and the environment variables. `SourceCode` uploads only the `pipeline_steps` folder and runs `train_fn.py`. The job runs on SageMaker managed infrastructure (`Mode.SAGEMAKER_TRAINING_JOB`).

In [ ]:
trainer_managed = ModelTrainer(
    training_image=xgboost_image,
    source_code=SourceCode(
        source_dir="pipeline_steps",
        entry_script="train_fn.py",
        ignore_patterns=["__pycache__", ".ipynb_checkpoints"],
    ),
    compute=Compute(
        instance_type=train_instance_type,
        instance_count=train_instance_count
    ),
    hyperparameters={
        **hyperparams,
        "output_s3_prefix": output_s3_prefix,
        "pipeline_run_id": preprocess_result["pipeline_run_id"],
    },
    base_job_name="from-idea-to-prod-training",
    training_mode=Mode.SAGEMAKER_TRAINING_JOB,
    output_data_config=OutputDataConfig(s3_output_path=output_s3_url),
    environment=job_environment,
)

In [ ]:
# train() waits until the job completes and raises an error if the job fails
trainer_managed.train(
    input_data_config=training_input_data,
    wait=True,
    logs=False,
)

The training job saved the model artifacts to S3.

In [ ]:
# ModelTrainer keeps the last training job it started. The SDK has no public accessor for it yet.
training_job = trainer_managed._latest_training_job
training_job_name = training_job.training_job_name

print(f"Training job:    {training_job_name}")
print(f"Model artifacts: {training_job.model_artifacts.s3_model_artifacts}")

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="top" href="https://{}.console.aws.amazon.com/sagemaker/home?region={}#/jobs/{}">training job</a></b>'.format(
            region, region, training_job_name))
)

#### Make the model deployable, then register it in the MLflow model registry
Because the MLflow App was created with `AutoModelRegistrationEnabled` (see notebook 00), every `mlflow.register_model()` call also automatically creates a corresponding Model Package in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html). You don't need to register in both places separately.

A Model Package is only *deployable* if it carries an **inference specification** — the serving container image, the model data location, and the environment variables the serving stack needs. So before registering, you:

1. Upload an `inference.py` into the logged model's artifact store under `code/`, using `MlflowClient.log_model_artifacts()`
2. Log an **inference specification** on the logged model with the `sagemaker-mlflow` plugin (`>= 0.5.0`)
3. Call `mlflow.register_model()` — the auto-sync copies the specification onto the Model Package

```
MLflow artifact store ◀────────── serves directly from ──────────┐
        │  log_inference_specification()                         │
        ▼                                                        │
MLflow ──register──▶ auto-sync ──▶ Model Package (deployable) ──▶ Endpoint
```

The MLflow artifact store stays the single source of truth for the model bytes: no repacking, no artifact copies.

<div class="alert alert-warning">
<b>Order matters:</b> the specification must be logged <b>before</b> <code>mlflow.register_model()</code>. The auto-sync copies the specification onto the Model Package at registration time — log it afterwards and the package is already created without it.
</div>

In [ ]:
# Reuse the single registered model name shared across all notebooks
# (created in notebook 01, restored via %store). New registrations become new
# VERSIONS of the same registered model instead of a new model per notebook.
if "registered_model_name" not in dir():
    import time
    registered_model_name = f"from-idea-to-prod-model-{time.strftime('%d%m%y', time.gmtime())}"
    %store registered_model_name

print(f"Registering to model: {registered_model_name}")

Find the MLflow run of the `ModelTrainer` job and resolve its **logged model**: the MLflow 3 entity that carries the `model_id` and `artifact_location` the inference specification points at. `train()` tags each run with the SageMaker job name, and it runs `mlflow.xgboost.autolog(..., log_models=False)` and logs the model explicitly with `mlflow.xgboost.log_model()`, which is what makes it a searchable logged model on that run.

In [ ]:
mlflow_client = MlflowClient()
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# pipeline_steps/runtime_utils.py tags every run with the SageMaker job name
training_runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=f"tags.`sagemaker.job_name` = '{training_job_name}'",
)
if training_runs.empty:
    raise RuntimeError(f"No MLflow run found for the training job {training_job_name}")
last_run_id = training_runs["run_id"][0]

# Resolve the MLflow 3 logged model produced by that run
logged_model = next(
    m for m in mlflow.search_logged_models(
        experiment_ids=[experiment_id], output_format="list"
    )
    if m.source_run_id == last_run_id
)

print(f"Source run:        {last_run_id}")
print(f"Logged model id:   {logged_model.model_id}")
print(f"Artifact location: {logged_model.artifact_location}")

#### Log an `inference.py` into the model's artifact store

The BYOC image serves with its own FastAPI stack (see [`byoc/serving_app.py`](byoc/serving_app.py)), which loads the script named by `SAGEMAKER_PROGRAM` and dispatches requests to `model_fn` / `input_fn` / `predict_fn` / `output_fn`. It provides no default model loader, so a `model_fn` is mandatory — without it the container fails every `/ping` health check and the endpoint never reaches `InService`.

The `S3Prefix` `ModelDataSource` in the specification downloads *everything* under the logged model's artifact location to `/opt/ml/model`, so a file logged at `code/inference.py` arrives at `/opt/ml/model/code/inference.py`, and the MLflow model files (`MLmodel`, the booster, `conda.yaml`) land at `/opt/ml/model` — which is why `model_fn` can just call `mlflow.xgboost.load_model(model_dir)`.

The handlers are in [`pipeline_steps/inference.py`](pipeline_steps/inference.py). The register step of the ML pipeline in step 3 logs the same file. The next cell logs it with MLflow instead of a direct S3 upload, so the artifact store stays the single source of truth and the script shows up in the MLflow UI next to the model.

In [ ]:
# log_model_artifacts keeps the directory layout, so code/inference.py
# lands at <artifact_location>/code/inference.py
with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, "code"))
    shutil.copy("pipeline_steps/inference.py", os.path.join(tmp, "code", "inference.py"))
    mlflow_client.log_model_artifacts(logged_model.model_id, tmp)

print(f"Logged code/inference.py to {logged_model.artifact_location}/code/")

#### Log the inference specification

`sagemaker_mlflow.log_inference_specification()` attaches the container image, the model data source, and the environment variables to the logged model. The instance types you list here constrain what the Model Package can be deployed on, so include the type you intend to use.

In [ ]:
inference_spec = {
    "Containers": [{
        "Image": xgboost_image,
        # Serve straight from the MLflow artifact store - no repacking, no copies.
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": logged_model.artifact_location.rstrip("/") + "/",
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        # Point the BYOC serving stack at the script logged above.
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        },
    }],
    "SupportedContentTypes": ["text/csv", "application/json"],
    "SupportedResponseMIMETypes": ["text/csv", "application/json"],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.m5.xlarge", "ml.r5.xlarge"],
    "SupportedTransformInstanceTypes": ["ml.m5.large", "ml.m5.xlarge"],
}

sagemaker_mlflow.log_inference_specification(
    logged_model.model_id, inference_specification=inference_spec
)
print("Inference specification logged on the MLflow model.")

#### Register, and watch the auto-sync

Now register the logged model. The sync to the SageMaker Model Registry runs asynchronously, and it records the resulting Model Package ARN as a `sagemaker.model_package_arn` **tag on the MLflow model version** — which is a much simpler way to find the package than searching the registry.

In [ ]:
registered_model_version = mlflow.register_model(
    f"models:/{logged_model.model_id}", registered_model_name
)
print(f"Registered in MLflow: {registered_model_version.name} v{registered_model_version.version}")

model_package_arn = None
for _ in range(20):
    mv = mlflow_client.get_model_version(
        registered_model_version.name, registered_model_version.version
    )
    model_package_arn = mv.tags.get("sagemaker.model_package_arn")
    if model_package_arn:
        break
    print(".", end="", flush=True)
    time.sleep(3)

if not model_package_arn:
    raise TimeoutError("SageMaker Model Package was not auto-created within the timeout")

print(f"\nSynced SageMaker Model Package: {model_package_arn}")

#### Inspect the auto-created Model Package

The package already carries an `InferenceSpecification` — that is the payoff of logging the spec *before* registering. The cell below reads it back from the registry and fails loudly if the environment variables did not survive the sync: without `SAGEMAKER_PROGRAM` the container would have no inference code to run.

In [ ]:
model_package = ModelPackage.get(model_package_name=model_package_arn)

# DescribeModelPackage does not return ModelPackageName for versioned packages,
# which breaks refresh()/update(). The API accepts an ARN in that field, so
# backfill it here.
model_package.model_package_name = model_package.model_package_arn

# The typed ContainersItem shape omits Environment and friends, so read the raw
# DescribeModelPackage response for those.
described = boto3.client("sagemaker").describe_model_package(
    ModelPackageName=model_package_arn
)
raw_container = described["InferenceSpecification"]["Containers"][0]

print(f"Model package group:  {model_package.model_package_group_name}")
print(f"Version:              {model_package.model_package_version}")
print(f"Status:               {model_package.model_package_status}")
print(f"Approval status:      {described.get('ModelApprovalStatus', '(not set)')}")

print("\nInference specification, as stored in the registry")
print(f"  image:              {raw_container['Image']}")
print(f"  model data source:  {raw_container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"  environment:        {raw_container.get('Environment', {})}")

expected_env = inference_spec["Containers"][0]["Environment"]
synced_env = raw_container.get("Environment", {})
mismatched = {k: (v, synced_env.get(k)) for k, v in expected_env.items() if synced_env.get(k) != v}
if mismatched:
    raise RuntimeError(
        f"Environment did not round-trip through the MLflow sync: {mismatched}. "
        "The endpoint would fall back to the container's defaults."
    )
print("\nEnvironment round-tripped through the sync intact.")

#### View the logged model in the MLflow UI

Open the MLflow UI with a presigned URL, then jump straight to the logged model's artifacts — you can see `code/inference.py` sitting next to the model files, exactly what the endpoint will download to `/opt/ml/model`.

In [ ]:
sm_client = boto3.Session().client("sagemaker")

experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# get the presigned url to open the MLflow UI
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

logged_model_artifacts = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/models/{logged_model.model_id}/artifacts?workspace=default"

In [ ]:
# Open the MLflow UI first: the presigned URL authenticates your browser session
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(presigned_url, quote=True)}">'
    '1. Open the MLflow UI and authenticate</a></b>'
))

In [ ]:
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(logged_model_artifacts, quote=True)}">'
    '2. Open the logged model artifacts in MLflow</a></b>'
))

Print the metrics and metadata of the registered model.

In [ ]:
print(f"Model: {registered_model_version.name}, version: {registered_model_version.version}")
print(f"Tags: {registered_model_version.tags}")
print(f"Aliases: {registered_model_version.aliases}")
print(f"\nMetrics:")
mlflow.get_run(logged_model.source_run_id).data.metrics

## Test the registered model locally
Before you move on, run a quick check of the registered model on the test set, directly in the notebook. Deploying the approved model to a real-time endpoint is part of [step 4](04-deployment-and-monitoring.ipynb), where you also add data and model monitoring.

### Prepare the test data
Download the prepared test dataset from S3 to the notebook.

In [ ]:
!aws s3 cp $test_s3_url/test_x.csv tmp/test_x.csv
!aws s3 cp $test_s3_url/test_y.csv tmp/test_y.csv

### Local inference in the notebook
First run inference locally using the registered model from the MLflow model registry.

In [ ]:
model_uri = registered_model_version.source
model = mlflow.xgboost.load_model(model_uri)

In [ ]:
test_x = np.loadtxt("tmp/test_x.csv", delimiter=",")
test_y = np.loadtxt("tmp/test_y.csv", delimiter=",")

dtest = xgb.DMatrix(test_x)
predictions = np.array(model.predict(dtest), dtype=float).squeeze()
predictions

In [ ]:
pd.crosstab(
    index=test_y,
    columns=np.round(predictions), 
    rownames=['actuals'], 
    colnames=['predictions']
)

#### Save charts to the MLflow run

It is a good idea to save the confusion matrices from the test data in MLflow for reference.

In [ ]:
def plot_confusion_matrix(
    cm, class_names, title="Confusion matrix", cmap=plt.cm.Blues, normalize=False
):
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        ylim=(cm.shape[0] - 0.5, -0.5),
        xticklabels=class_names,
        yticklabels=class_names,
        title=title,
        ylabel="Ground truth label",
        xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    fmt = ".2f"
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax, fig

In [ ]:
class_names = ["no", "yes"]
confusion_matrix = metrics.confusion_matrix(test_y, np.round(predictions))
ax, fig = plot_confusion_matrix(confusion_matrix, class_names)

In [ ]:
print(f"Log confusion matrix to the model {registered_model_version.name} version {registered_model_version.version}")
mlflow.set_experiment(experiment_name)
with mlflow.start_run(run_id=logged_model.source_run_id):
    mlflow.log_figure(fig, "confusion_matrix.png")

## Explore experiments and model registry with the MLflow UI
You can see all logged metrics, parameters, and artifacts in the MLflow UI. To launch the MLflow UI, choose **MLflow** in the **Application** pane of the Studio UI, select your MLflow server and choose **Open MLflow**:

Alternatively, run the next cell and click the link to open the MLflow UI directly from the notebook.

![](img/mlflow-open.png)

In the MLflow UI you can browse experiments, runs, and registered models:

![](img/experiment-mlflow-2.png)

![](img/models-mlflow-2.png)

![](img/model-fig-mlflow.png)

In [ ]:
# The presigned URL created earlier has been redeemed, so create a fresh one
presigned_url = boto3.Session().client("sagemaker").create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(presigned_url, quote=True)}">'
    'Open the MLflow UI</a></b>'
))

## Summary

In this notebook you moved from local notebook code to SageMaker managed infrastructure:

- **Reusable functions**: Moved the processing and training code into the `preprocess()` and `train()` functions in `pipeline_steps`, which the ML pipeline in step 3 reuses
- **Processing with `@remote`**: Validated `preprocess()` locally, then ran the same function as a SageMaker job
- **Training with `@remote`**: Validated `train()` locally, then ran the same function as a SageMaker training job
- **Training with `ModelTrainer`**: Ran the same `train()` code in script mode, with full control over the training container, input channels, and artifacts
- **Registration**: Logged an inference specification on the MLflow model *before* registering it, so the auto-synced SageMaker Model Package arrived deployable
- **Local test**: Scored the registered model on the test set and logged the confusion matrix to MLflow

In the next notebook, you'll combine the same functions into an **automated ML pipeline** using SageMaker Pipelines.

## Further development ideas for your real-world projects
- Learn more about [managing experiments with MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- Try no-code [SageMaker Canvas](https://docs.aws.amazon.com/sagemaker/latest/dg/canvas.html) on your data to perform analysis and use automated ML to build models and generate predictions.
- To improve the model training, try [automated hyperparameter tuning](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html).

## Additional resources
- [Using Docker containers with SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html)
- [Amazon SageMaker Immersion Day](https://catalog.us-east-1.prod.workshops.aws/workshops/63069e26-921c-4ce1-9cc7-dd882ff62575/en-US)
- [Amazon SageMaker Examples GitHub repository](https://github.com/aws/amazon-sagemaker-examples)
- [Amazon SageMaker Community Examples GitHub repository](https://github.com/aws/amazon-sagemaker-examples-community)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>